In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.optim as optim
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import time
torch.autograd.set_detect_anomaly(False)
torch.manual_seed(18)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

Using device: mps


In [2]:
class NeuralNet(nn.Module):
    def __init__(self,input_dimension,output_dimension,n_hidden_layers,neurons,regularization_param,regularization_exp,retrain_seed):
        super(NeuralNet, self).__init__()
        self.input_dimension = input_dimension
        self.output_dimension = output_dimension
        self.neurons = neurons
        self.n_hidden_layers = n_hidden_layers
        self.activation = nn.Tanh()
        self.regularization_param = regularization_param
        self.regularization_exp = regularization_exp
        self.input_layer = nn.Linear(self.input_dimension,self.neurons)
        self.hidden_layers = nn.ModuleList([nn.Linear(self.neurons,self.neurons) for _ in range(n_hidden_layers -1)])
        self.output_layer =  nn.Linear(self.neurons, self.output_dimension)
        self.retrain_seed = retrain_seed
        self.init_xavier()
    def forward(self,x):
        x = self.activation(self.input_layer(x))
        for k,l in enumerate(self.hidden_layers):
            x = self.activation(l(x))
        return self.output_layer(x)
    def init_xavier(self):
        torch.manual_seed(self.retrain_seed)
        def init_weights(m):
            if type(m) == nn.Linear and m.weight.requires_grad and m.bias.requires_grad:
                 g = nn.init.calculate_gain("tanh")
                 torch.nn.init.xavier_uniform_(m.weight, gain = g)
                 m.bias.data.fill_(0)
        self.apply(init_weights)
    
    def regularization(self):
        reg_loss = 0
        for name, param in self.named_parameters():
            if 'weight' in name:
                reg_loss =  reg_loss+torch.norm(param,self.regularization_exp)
        return self.regularization_param * reg_loss

In [ ]:
class Pinns:
    def __init__(self, n_int_, n_sb_, n_tb_,device = device):
        self.n_int = n_int_
        self.n_sb = n_sb_
        self.n_tb = n_tb_
        self.device = device
        self.domain_extrema = torch.tensor([[0,1],[-1,1],[-1,1]])
        self.space_dimensions = 2
        self.lambda_u = 10
        self.approximate_solution = NeuralNet(input_dimension = self.domain_extrema.shape[0],output_dimension =1,
                                              n_hidden_layers = 4,
                                              neurons = 20,
                                              regularization_param = 0.,
                                              regularization_exp = 2.,
                                              retrain_seed = 42).to(self.device)
        self.soboleng = torch.quasirandom.SobolEngine(dimension= self.domain_extrema.shape[0])
        self.training_set_sb, self.training_set_tb, self.training_set_int = self.assemble_datasets()
    def convert(self,tens):
        assert(tens.shape[1] == self.domain_extrema.shape[0])
        return tens*(self.domain_extrema[:,1] - self.domain_extrema[:,0]) + self.domain_extrema[:,0]
    def initial_condition(self,x):
 
        return torch.zeros(x.shape[0])

  
    def add_temporal_boundary_points(self):
       
     
    def add_spatial_boundary_points(self):
        return 
      
       
    def add_interior_points(self):
        return 
        
    def assemble_datasets(self):
        return 
        
        
    def apply_initial_condition(self):
        return 
        
    def apply_boundary_conditions(self):
        return 
      
    def compute_pde_residual(self):
        return 
       
    def compute_loss(self):
       
    def fit(self, num_epochs, optimizer, verbose=True):
        history = list()

        # Loop over epochs
        for epoch in range(num_epochs):
            if verbose: print("################################ ", epoch, " ################################")

            for j, ((inp_train_sb, u_train_sb), (inp_train_tb, u_train_tb), (inp_train_int, u_train_int)) in enumerate(zip(self.training_set_sb, self.training_set_tb, self.training_set_int)):
                def closure():
                    optimizer.zero_grad()
                    loss = self.compute_loss(inp_train_sb, u_train_sb, inp_train_tb, u_train_tb, inp_train_int, verbose=verbose)
                    loss.backward()

                    history.append(loss.item())
                    return loss

                optimizer.step(closure=closure)

        print('Final Loss: ', history[-1])

        return history
    def plotting(self):
